<a href="https://colab.research.google.com/github/abdul-sgf3/ProyekAkhir_5A_Adult_Census_Income/blob/main/02.Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing – Adult Census Income

Notebook ini khusus untuk **preprocessing** dataset `adult.data`, terpisah dari notebook EDA.

Tahapan:
1. Membaca dataset dan memberi nama kolom.
2. Mengubah tanda `?` menjadi missing value.
3. Menghapus data duplikat.
4. Mengisi missing value numerik dengan median dan kategorikal dengan modus.
5. Mengubah fitur kategorikal menggunakan One-Hot Encoding.
6. Melakukan standardisasi fitur numerik.
7. Mengubah target `income` menjadi 0 dan 1.
8. Menyimpan hasil menjadi `adult_preprocessed.csv`.



In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [2]:
from google.colab import files
uploaded = files.upload()  # Pilih file adult.data

FILE_PATH = "adult.data"  # Sesuaikan jika nama file berbeda
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

df = pd.read_csv(FILE_PATH, header=None, names=columns, skipinitialspace=True)
print("Ukuran data awal:", df.shape)
display(df.head())

Saving adult.data to adult.data
Ukuran data awal: (32561, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
# 1. Membersihkan spasi dan mengubah tanda ? menjadi NaN
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df = df.replace("?", np.nan)
print("Missing value sebelum penanganan:")
display(df.isna().sum().sort_values(ascending=False))

Missing value sebelum penanganan:


,0
occupation,1843
workclass,1836
native_country,583
fnlwgt,0
education,0
education_num,0
age,0
marital_status,0
relationship,0
sex,0


In [4]:
# 2. Menghapus baris duplikat
jumlah_duplikat = df.duplicated().sum()
df = df.drop_duplicates().copy()
print("Duplikat yang dihapus:", jumlah_duplikat)
print("Ukuran setelah hapus duplikat:", df.shape)

Duplikat yang dihapus: 24
Ukuran setelah hapus duplikat: (32537, 15)


In [5]:
# 3. Memisahkan fitur dan target
X = df.drop(columns=["income"])
y = df["income"].str.strip()

# Target: <=50K menjadi 0 dan >50K menjadi 1
# Pastikan label sesuai dengan isi dataset
print("Kategori target:", y.value_counts(dropna=False))

Kategori target: income
<=50K    24698
>50K      7839
Name: count, dtype: int64


In [6]:
# 4. Menentukan kolom numerik dan kategorikal
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)

Fitur numerik: ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
Fitur kategorikal: ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']


In [12]:
# 5. Membuat pipeline preprocessing
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [8]:
# 6. Menjalankan preprocessing
X_processed = preprocessor.fit_transform(X)

# Mengambil nama kolom hasil transformasi
feature_names = preprocessor.get_feature_names_out()
X_processed_df = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=X.index
)

# Mengubah target menjadi angka
y_processed = y.map({"<=50K": 0, ">50K": 1})

# Periksa apakah ada label target yang tidak dikenali
if y_processed.isna().any():
    raise ValueError("Ada kategori income yang tidak dikenali. Periksa kembali nilai pada kolom income.")

y_processed = y_processed.astype(int)
processed_df = X_processed_df.copy()
processed_df["income"] = y_processed

print("Ukuran data setelah preprocessing:", processed_df.shape)
display(processed_df.head())

Ukuran data setelah preprocessing: (32537, 106)


,num__age,num__fnlwgt,num__education_num,num__capital_gain,num__capital_loss,num__hours_per_week,cat__workclass_Federal-gov,cat__workclass_Local-gov,cat__workclass_Never-worked,cat__workclass_Private,...,cat__native_country_Puerto-Rico,cat__native_country_Scotland,cat__native_country_South,cat__native_country_Taiwan,cat__native_country_Thailand,cat__native_country_Trinadad&Tobago,cat__native_country_United-States,cat__native_country_Vietnam,cat__native_country_Yugoslavia,income
0,0.030390,-1.063569,1.134777,0.148292,-0.216743,-0.035664,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
1,0.836973,-1.008668,1.134777,-0.145975,-0.216743,-2.222483,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
2,-0.042936,0.245040,-0.420679,-0.145975,-0.216743,-0.035664,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
3,1.056950,0.425752,-1.198407,-0.145975,-0.216743,-0.035664,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0
4,-0.776193,1.408066,1.134777,-0.145975,-0.216743,-0.035664,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [9]:
# 7. Memastikan hasil preprocessing bersih
print("Total missing value:", processed_df.isna().sum().sum())
print("Jumlah duplikat:", processed_df.duplicated().sum())
print("Distribusi target:")
display(processed_df["income"].value_counts())

Total missing value: 0
Jumlah duplikat: 0
Distribusi target:


,count
income,
0,24698
1,7839


In [10]:
# 8. Menyimpan hasil preprocessing
OUTPUT_PATH = "adult_preprocessed.csv"
processed_df.to_csv(OUTPUT_PATH, index=False)
print(f"File berhasil disimpan: {OUTPUT_PATH}")

# Download hasil ke komputer (Google Colab)
files.download(OUTPUT_PATH)

File berhasil disimpan: adult_preprocessed.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Keterangan hasil

- `income = 0` berarti pendapatan `<=50K`.
- `income = 1` berarti pendapatan `>50K`.
- Kolom kategorikal sudah diubah menjadi kolom numerik melalui One-Hot Encoding.
- Fitur numerik sudah distandardisasi.
- File `adult_preprocessed.csv` dapat digunakan sebagai input untuk tahap pemodelan machine learning.

**Catatan:** preprocessing di atas melakukan `fit_transform` pada seluruh data agar menghasilkan satu file hasil preprocessing. Jika nanti melakukan evaluasi model, pembagian train-test sebaiknya dilakukan terlebih dahulu dan proses preprocessing di-fit hanya pada data training untuk menghindari data leakage.